<a href="https://colab.research.google.com/github/YifeiCAO/CogModelingRNNsTutorial/blob/main/three_arm_hybridrnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import and preparation

In [ ]:
AS_OUTER_LOOP = [1,2,3,4,5,6,7,8,9,10] # assigned outer loop(s) to run in this notebook

In [ ]:
!pip -q install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.8 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd
from io import StringIO
import gdown

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import numpy as np
import pandas as pd
# import plotnine as gg
# gg.theme_set(gg.theme_classic)  # for nicer-looking plots
import jax.numpy as jnp
import jax
import optax
import scipy

!pip install -U dm-haiku
!pip install chex
import haiku as hk
rng_seq = hk.PRNGSequence(np.random.randint(2**32))

#@title Install required packages.
try:
    from google.colab import files
    _ON_COLAB = True
except:
    _ON_COLAB = False

if _ON_COLAB:
  !rm -rf CogModelingRNNsTutorial
  !git clone -b train https://github.com/YifeiCAO/CogModelingRNNsTutorial
  !pip install -e CogModelingRNNsTutorial/CogModelingRNNsTutorial
  !cp CogModelingRNNsTutorial/CogModelingRNNsTutorial/*py CogModelingRNNsTutorial
else:
  !pip install CogModelingRNNsTutorial/requirements.txt

#@title Imports + defaults settings.
# %load_ext autoreload
# %autoreload 2
# for reload
# %reload_ext autoreload

# import haiku as hk
# import jax
# import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optax
import os
import warnings

# warnings.filterwarnings("ignore")

# try:
#     from google.colab import files
#     _ON_COLAB = True
# except:
#     _ON_COLAB = False

from CogModelingRNNsTutorial import bandits
from CogModelingRNNsTutorial import disrnn
from CogModelingRNNsTutorial import hybrnn
from CogModelingRNNsTutorial import hybrnn_twostep
from CogModelingRNNsTutorial import hybconrnn
from CogModelingRNNsTutorial import hybrnn_direct_con
from CogModelingRNNsTutorial import plotting
from CogModelingRNNsTutorial import rat_data
from CogModelingRNNsTutorial import rnn_utils

from google.colab import drive
import pickle

# 挂载 Google Drive
drive.mount('/content/drive')

Cloning into 'CogModelingRNNsTutorial'...
remote: Enumerating objects: 1379, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 1379 (delta 213), reused 192 (delta 161), pack-reused 1112 (from 1)
Receiving objects: 100% (1379/1379), 6.88 MiB | 33.41 MiB/s, done.
Resolving deltas: 100% (886/886), done.
Obtaining file:///content/CogModelingRNNsTutorial/CogModelingRNNsTutorial
  Preparing metadata (setup.py) ... done
  Running setup.py develop for CogModelingRNNsTutorial
Mounted at /content/drive


In [ ]:
def compute_negative_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]

    # 直接在外部运行 eval_model（不要放 jit 里！）
    # rnn_utils.eval_model 内已用 jnp.stack 得到 (T, N, C)，与 fit_model 一致
    model_outputs, _ = rnn_utils.eval_model(model_fun, params, xs)

    @jax.jit
    def compute_log_probs(logits):
        return jax.nn.log_softmax(logits, axis=-1)

    predicted_log_choice_probabilities = compute_log_probs(jnp.asarray(model_outputs))

    # 后续仍转为 NumPy 做索引
    predicted_log_choice_probabilities = np.array(predicted_log_choice_probabilities)
    n_actions = predicted_log_choice_probabilities.shape[2]

    total_log_likelihood = 0.0
    total_valid_trials = 0

    for sess_i in range(n_sessions):
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            if 0 <= actual_choice < n_actions:
                total_log_likelihood += predicted_log_choice_probabilities[trial_i, sess_i, actual_choice]
                total_valid_trials += 1

    if total_valid_trials == 0:
        raise ValueError("No valid trials found.")

    avg_nll = -total_log_likelihood / total_valid_trials
    return avg_nll



def evaluate_nll_over_full_dataset(dataset, model_fun, params):
    all_nlls = []
    for _ in range(dataset.n_batches):
        nll = compute_negative_log_likelihood(dataset, model_fun, params)
        all_nlls.append(nll)

    avg_nll = np.mean(all_nlls)
    print(f"[All Batches Averaged] NLL: {avg_nll:.4f}")
    return avg_nll


In [ ]:
@jax.jit
def compute_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]
    model_outputs, model_states = rnn_utils.eval_model(model_fun, params, xs)

    # (T, N, C)，对最后一维做 softmax
    predicted_log_choice_probabilities = np.array(
        jax.nn.log_softmax(jnp.asarray(model_outputs), axis=-1)
    )

    n_actions = predicted_log_choice_probabilities.shape[2]
    log_likelihoods = []

    for sess_i in range(n_sessions):
        log_likelihood = 0.0
        n = 0
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            # ignore invalid trials (<0 or ≥n_actions)
            if 0 <= actual_choice < n_actions:
                log_likelihood += predicted_log_choice_probabilities[
                    trial_i, sess_i, actual_choice
                ]
                n += 1

        if n > 0:
            normalized_likelihood = np.exp(log_likelihood / n)
            log_likelihoods.append(normalized_likelihood)

    mean_likelihood = np.mean(log_likelihoods)
    std_likelihood  = np.std(log_likelihoods)

    print(f'Average Normalized Likelihood: {100 * mean_likelihood:.1f}%')
    return mean_likelihood, std_likelihood


In [ ]:
jax.devices()

[CpuDevice(id=0)]

# Human data (two steps)

## Shahar data

In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1eUdMZ1_HHo7Uw5LcZDVGGV_aRNVxQT14'
download_url = f'https://drive.google.com/uc?id={file_id}'


# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
shahar_data = pd.read_csv(output_file)

# Display the first few rows
shahar_data.head()

In [ ]:
import numpy as np

# 1. 创建唯一的被试ID (基于 subject 和 measurment)
shahar_data['unique_id'] = shahar_data.groupby(['subject', 'measurment']).ngroup()

# 2. 确保数据按被试和试次顺序排序
shahar_data = shahar_data.sort_values(['unique_id', 'trial']).reset_index(drop=True)

# 3. 生成下一步的动作 (action_n)，并在每个被试最后 trial 填补 -1
shahar_data['action_n'] = shahar_data.groupby('unique_id')['choice1'].shift(-1).fillna(-1).astype(int)

xs_list_sha = []
ys_list_sha = []
seg_len = 201

# 4. 按 unique_id 遍历并构造 padding
for pid, grp in shahar_data.groupby('unique_id'):
    # 特征包含 choice1, reward, transition 三列
    x_seq = grp[['choice1', 'reward', 'transition']].to_numpy().astype(float)
    y_seq = grp[['action_n']].to_numpy().astype(int)

    # 计算需要 pad 的长度
    pad_len = seg_len - len(x_seq)

    if pad_len > 0:
        # xs pad with 0.
        x_seq = np.pad(x_seq, pad_width=((0, pad_len), (0, 0)), constant_values=0.)
        # ys pad with -1
        y_seq = np.pad(y_seq, pad_width=((0, pad_len), (0, 0)), constant_values=-1)
    elif pad_len < 0:
        # 截断（如果多于201的情况，以防万一）
        x_seq = x_seq[:seg_len]
        y_seq = y_seq[:seg_len]

    xs_list_sha.append(x_seq)
    ys_list_sha.append(y_seq)

# 5. 堆叠成三维数组 (n_sessions, 201, feat_dim)
xs_sha_stacked = np.stack(xs_list_sha, axis=0)
ys_sha_stacked = np.stack(ys_list_sha, axis=0)

# 6. 转置为模型需要的维度: (n_trials, n_sessions, feat_dim) -> (201, N, 3 / 1)
xs_sha = xs_sha_stacked.transpose(1, 0, 2)
ys_sha = ys_sha_stacked.transpose(1, 0, 2)

print(f"xs_sha.shape = {xs_sha.shape}")
print(f"ys_sha.shape = {ys_sha.shape}")


## eisenberg data

In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1Qqrs20pf45SyB6HatfOnXFf4qwG53zFe'
download_url = f'https://drive.google.com/uc?id={file_id}'


# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
eisenberg_data = pd.read_csv(output_file)

# Display the first few rows
eisenberg_data.head()

In [ ]:
import numpy as np
import pandas as pd

# 1. 提取被试编号
eisenberg_data['subject'] = eisenberg_data['Unnamed: 0'].str.split('_').str[3]

# 2. 扔掉 practice 的行
eisenberg_data = eisenberg_data[eisenberg_data['exp_stage'] != 'practice'].reset_index(drop=True)

# 3. 把 stage_transition 里的 frequent 编码成 0，另一个编码成 1
eisenberg_data['stage_transition'] = np.where(eisenberg_data['stage_transition'] == 'frequent', 0, 1)

# 将所需的特征列转为合适类型，处理第一试次可能的 NaN
eisenberg_data['stim_selected_first'] = eisenberg_data['stim_selected_first'].astype(int)
eisenberg_data['feedback_last'] = eisenberg_data['feedback_last'].fillna(0).astype(float)

# 4. 生成 unique_id 并排序
eisenberg_data['unique_id'] = eisenberg_data.groupby('subject').ngroup()
eisenberg_data = eisenberg_data.sort_values(['unique_id', 'trial_num']).reset_index(drop=True)

# 5. 生成 action_n
eisenberg_data['action_n'] = eisenberg_data.groupby('unique_id')['stim_selected_first'].shift(-1).fillna(-1).astype(int)

xs_list_eisen = []
ys_list_eisen = []
seg_len = 201

# 6. 按 unique_id 遍历并构造 padding
for pid, grp in eisenberg_data.groupby('unique_id'):
    x_seq = grp[['stim_selected_first', 'feedback_last', 'stage_transition']].to_numpy().astype(float)
    y_seq = grp[['action_n']].to_numpy().astype(int)

    pad_len = seg_len - len(x_seq)

    if pad_len > 0:
        # xs pad with 0.
        x_seq = np.pad(x_seq, pad_width=((0, pad_len), (0, 0)), constant_values=0.)
        # ys pad with -1
        y_seq = np.pad(y_seq, pad_width=((0, pad_len), (0, 0)), constant_values=-1)
    elif pad_len < 0:
        # 截断（如果多于201的情况，以防万一）
        x_seq = x_seq[:seg_len]
        y_seq = y_seq[:seg_len]

    xs_list_eisen.append(x_seq)
    ys_list_eisen.append(y_seq)

# 7. 堆叠成三维数组
xs_eisen_stacked = np.stack(xs_list_eisen, axis=0)
ys_eisen_stacked = np.stack(ys_list_eisen, axis=0)

# 8. 转置为模型需要的维度: (n_trials, n_sessions, feat_dim)
xs_eisen = xs_eisen_stacked.transpose(1, 0, 2)
ys_eisen = ys_eisen_stacked.transpose(1, 0, 2)

print(f"xs_eisen.shape = {xs_eisen.shape}")
print(f"ys_eisen.shape = {ys_eisen.shape}")


# Training

In [ ]:
xs_human_list = [xs_sha, xs_eisen]
ys_human_list = [ys_sha, ys_eisen]

## RNN

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold
import haiku as hk

# RNN 核心：'vanilla' | 'gru' | 'lstm'（均为 hk.RNNCore，可与 hk.DeepRNN 串联）
RNN_CORE_TYPE = 'vanilla'

def _make_rnn_core(hidden_size: int):
    t = (RNN_CORE_TYPE or 'vanilla').lower().strip()
    if t == 'vanilla':
        return hk.VanillaRNN(hidden_size)
    if t == 'gru':
        return hk.GRU(hidden_size)
    if t == 'lstm':
        return hk.LSTM(hidden_size)
    raise ValueError("RNN_CORE_TYPE 须为 'vanilla'、'gru' 或 'lstm'")

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of arrays（此处为 Suthana + Barnby 两个实验）
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")
print(f"RNN core (Haiku): {RNN_CORE_TYPE}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [16, 32, 64, 128],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [128, 256],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results/twostep'
os.makedirs(OUTPUT_DIR, exist_ok=True)
_rnn_tag = (RNN_CORE_TYPE or 'vanilla').lower().strip()
inner_csv = os.path.join(OUTPUT_DIR, f'rnn_{_rnn_tag}_inner_search_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, f'rnn_{_rnn_tag}_outer_results.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx :', test_idx)

    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            def make_rnn():
                return hk.DeepRNN([_make_rnn_core(hs), hk.Linear(output_size=2)])

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_rnn,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True,
                if_mean=True,
                reduce_host_sync=True,
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_rnn, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    def make_best_rnn():
        return hk.DeepRNN([_make_rnn_core(best_params['hidden_size']), hk.Linear(output_size=2)])

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_rnn,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True,
        if_mean=True,
        reduce_host_sync=True,
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_rnn, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


## RL-ANN

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64, 128],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [128, 256],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results/twostep'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'rlann_inner_search_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'rlann_outer_results.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'False'
            use_previous_values = 'False'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_rlann():
              model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_rlann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True,
                if_mean=True,
                reduce_host_sync=True,
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_rlann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'False'
    use_previous_values = 'False'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_rlann():
      model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_rlann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True,
        if_mean=True,
        reduce_host_sync=True,
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_rlann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


## Context-ANN

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64, 128],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [128, 256],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results/twostep'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'conann_inner_search_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'conann_outer_results.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'False'
            use_previous_values = 'True'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_conann():
              model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_conann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True,
                if_mean=True,
                reduce_host_sync=True,
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_conann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'False'
    use_previous_values = 'True'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_conann():
      model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_conann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True,
        if_mean=True,
        reduce_host_sync=True,
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_conann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


## Memory-ANN

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 合并所有实验到 session 维
# =====================================
# 假设 xs_human_list / ys_human_list: list of 3 arrays
# 每个数组形状: (n_trials, n_sessions_exp, feature_dim)
xs_all = np.concatenate(xs_human_list, axis=1)  # 在 session 维拼接
ys_all = np.concatenate(ys_human_list, axis=1)  # 同样在 session 维拼接

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs,
                                  ys,
                                  dataset_constructor,
                                  train_idx,
                                  val_idx,
                                  test_idx,
                                  batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格
# =====================================
param_grid = {
    'hidden_size': [8, 16, 32, 64, 128],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [128, 256],
}
K_outer = 10
K_inner = 3
random_seed = 42

# =====================================
# Step 3: 结果保存路径
# =====================================
OUTPUT_DIR = '/content/drive/MyDrive/Results/twostep'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'memann_inner_search_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'memann_outer_results.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'hidden_size', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_hidden_size', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

# =====================================
# Step 4: 外层 CV
# =====================================
outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        # print('train_val_idx: ', train_val_idx)
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    # ==================
    # 内层超参搜索
    # ==================
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for hs, lr, wd, bs in itertools.product(
            param_grid['hidden_size'],
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['hidden_size'] == hs) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: hs={hs}, lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):



            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            use_hidden_state = 'True'
            use_previous_values = 'False'
            fit_forget = "False"
            habit_weight = 1

            value_weight = 1.

            rnn_rl_params = {
                's': use_hidden_state == 'True',
                'o': use_previous_values == 'True',
                'fit_forget': fit_forget == 'True',
                'forget': 0.,
                'w_h': habit_weight,
                'w_v': value_weight}
            network_params = {'n_actions': 2, 'hidden_size': hs}

            def make_memann():
              model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
              return model

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_memann,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True,
                if_mean=True,
                reduce_host_sync=True,
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_memann, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'hidden_size': hs, 'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    # ==================
    # 外层训练 + 测试
    # ==================
    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    use_hidden_state = 'True'
    use_previous_values = 'False'
    fit_forget = "False"
    habit_weight = 1

    value_weight = 1.

    rnn_rl_params = {
        's': use_hidden_state == 'True',
        'o': use_previous_values == 'True',
        'fit_forget': fit_forget == 'True',
        'forget': 0.,
        'w_h': habit_weight,
        'w_v': value_weight}
    network_params = {'n_actions': 2, 'hidden_size': best_params['hidden_size']}

    def make_best_memann():
      model = hybrnn_twostep.BiRNN(rl_params=rnn_rl_params, network_params=network_params)
      return model

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_memann,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True,
        if_mean=True,
        reduce_host_sync=True,
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_memann, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_hidden_size': best_params['hidden_size'],
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

# =====================================
# Step 5: 最终结果
# =====================================
avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")


## RL

In [ ]:
import numpy as np
import itertools
import os
import pandas as pd
import time
from sklearn.model_selection import KFold

# =====================================
# Step 0: 与 RNN / RL-ANN 相同的人类数据与标签
# =====================================
xs_all = np.concatenate(xs_human_list, axis=1)
ys_all = np.concatenate(ys_human_list, axis=1)

n_trials, total_sessions, feature_dim = xs_all.shape
_, _, target_dim = ys_all.shape
N_ACTIONS_RL = 2

print(f"总 trials: {n_trials}, 总 sessions: {total_sessions}")
print(f"X shape: {xs_all.shape}, Y shape: {ys_all.shape}")

# =====================================
# Step 1: 按索引切分 dataset
# =====================================
def format_into_datasets_by_index(xs, ys, dataset_constructor,
                                  train_idx, val_idx, test_idx, batch_size=None):
    ds_train = dataset_constructor(xs[:, train_idx], ys[:, train_idx], batch_size=batch_size)
    ds_val   = dataset_constructor(xs[:, val_idx], ys[:, val_idx], batch_size=batch_size)
    ds_test  = dataset_constructor(xs[:, test_idx], ys[:, test_idx], batch_size=batch_size)
    return ds_train, ds_val, ds_test

# =====================================
# Step 2: 超参数网格（认知 RL 无 hidden_size；alpha/beta 由 fit 学习）
# =====================================
param_grid = {
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'batch_size': [128, 256],
}
K_outer = 10
K_inner = 3
random_seed = 42

OUTPUT_DIR = '/content/drive/MyDrive/Results/twostep'
os.makedirs(OUTPUT_DIR, exist_ok=True)
inner_csv = os.path.join(OUTPUT_DIR, 'preserve_con_rl_inner_search_results.csv')
outer_csv = os.path.join(OUTPUT_DIR, 'preserve_con_rl_outer_results.csv')

if os.path.exists(inner_csv):
    df_inner_existing = pd.read_csv(inner_csv)
else:
    df_inner_existing = pd.DataFrame(columns=[
        'outer_fold', 'lr', 'weight_decay', 'batch_size', 'avg_val_nll', 'time'
    ])
if os.path.exists(outer_csv):
    df_outer_existing = pd.read_csv(outer_csv)
else:
    df_outer_existing = pd.DataFrame(columns=[
        'outer_fold', 'best_lr', 'best_weight_decay', 'best_batch_size', 'test_nll'
    ])

outer_cv = KFold(n_splits=K_outer, shuffle=True, random_state=random_seed)
session_indices = np.arange(total_sessions)

for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(session_indices), start=1):

    print(f"\n=== Outer Fold {outer_fold}/{K_outer} ===")

    if outer_fold not in AS_OUTER_LOOP:
        print(f"Skipping outer fold {outer_fold} (not assigned in this notebook)")
        print('test_idx: ', test_idx)
        continue

    if (df_outer_existing['outer_fold'] == outer_fold).any():
        print(f"Skipping outer fold {outer_fold} (already done)")
        continue

    print('test_idx: ', test_idx)
    inner_cv = KFold(n_splits=K_inner, shuffle=True, random_state=random_seed)
    best_score = float('inf')
    best_params = None

    for lr, wd, bs in itertools.product(
            param_grid['lr'],
            param_grid['weight_decay'],
            param_grid['batch_size']):

        mask = (
            (df_inner_existing['outer_fold'] == outer_fold) &
            (df_inner_existing['lr'] == lr) &
            (df_inner_existing['weight_decay'] == wd) &
            (df_inner_existing['batch_size'] == bs)
        )
        if mask.any():
            print(f"Skipping inner config fold {outer_fold}: lr={lr}, wd={wd}, bs={bs}")
            continue

        inner_scores = []
        t0 = time.time()

        for inner_train_idx, inner_val_idx in inner_cv.split(train_val_idx):

            train_sessions = train_val_idx[inner_train_idx]
            val_sessions = train_val_idx[inner_val_idx]

            ds_train, ds_val, _ = format_into_datasets_by_index(
                xs_all, ys_all,
                rnn_utils.DatasetRNN,
                train_idx=train_sessions,
                val_idx=val_sessions,
                test_idx=[],
                batch_size=bs
            )

            def make_cog_rl():
                return bandits.Hk_PreserveConAgentQ(n_actions=N_ACTIONS_RL, reward_col=1)

            params, _, _ = rnn_utils.fit_model(
                model_fun=make_cog_rl,
                dataset_train=ds_train,
                dataset_test=ds_val,
                loss_fun='categorical',
                optimizer=optax.chain(
                    optax.add_decayed_weights(wd),
                    optax.adam(learning_rate=lr)
                ),
                n_steps_per_call=1000,
                n_steps_max=100000,
                early_stop_step=200,
                if_early_stop=True,
                if_mean=True,
                reduce_host_sync=True,
            )

            avg_nll = evaluate_nll_over_full_dataset(ds_val, make_cog_rl, params)
            inner_scores.append(avg_nll)

        mean_inner_score = np.mean(inner_scores)
        df_inner_existing = pd.concat([
            df_inner_existing,
            pd.DataFrame([{
                'outer_fold': outer_fold,
                'lr': lr, 'weight_decay': wd, 'batch_size': bs,
                'avg_val_nll': mean_inner_score,
                'time': time.time() - t0
            }])
        ], ignore_index=True)
        df_inner_existing.to_csv(inner_csv, index=False)

        if mean_inner_score < best_score:
            best_score = mean_inner_score
            best_params = {'lr': lr, 'weight_decay': wd, 'batch_size': bs}

    print(f"Best params for outer fold {outer_fold}: {best_params}, val NLL={best_score:.4f}")

    ds_train_val, _, ds_test = format_into_datasets_by_index(
        xs_all, ys_all,
        rnn_utils.DatasetRNN,
        train_idx=train_val_idx,
        val_idx=[],
        test_idx=test_idx,
        batch_size=best_params['batch_size']
    )

    def make_best_cog_rl():
        return bandits.Hk_PreserveConAgentQ(n_actions=N_ACTIONS_RL, reward_col=1)

    params, _, _ = rnn_utils.fit_model(
        model_fun=make_best_cog_rl,
        dataset_train=ds_train_val,
        dataset_test=ds_test,
        loss_fun='categorical',
        optimizer=optax.chain(
            optax.add_decayed_weights(best_params['weight_decay']),
            optax.adam(learning_rate=best_params['lr'])
        ),
        n_steps_per_call=1000,
        n_steps_max=100000,
        early_stop_step=200,
        if_early_stop=True,
        if_mean=True,
        reduce_host_sync=True,
    )

    test_nll = evaluate_nll_over_full_dataset(ds_test, make_best_cog_rl, params)
    print(f"Outer fold {outer_fold} test NLL: {test_nll:.4f}")

    df_outer_existing = pd.concat([
        df_outer_existing,
        pd.DataFrame([{
            'outer_fold': outer_fold,
            'best_lr': best_params['lr'],
            'best_weight_decay': best_params['weight_decay'],
            'best_batch_size': best_params['batch_size'],
            'test_nll': test_nll
        }])
    ], ignore_index=True)
    df_outer_existing.to_csv(outer_csv, index=False)

avg_nll = df_outer_existing['test_nll'].mean()
se_nll = df_outer_existing['test_nll'].std(ddof=1) / np.sqrt(len(df_outer_existing))
print(f"\nFinal Average NLL: {avg_nll:.4f} ± {se_nll:.4f}")
